# Generate Synthetic Cast Alloy Database

Uses GMM (tuned in 02 for cast) to sample compositions and per-target forward models (from config) to predict properties. Saves **synthetic_cast.csv** as the search pool for 05_backward_cast. Run 01 and 02 first.

In [1]:
import pandas as pd
import numpy as np
import re
import os
import xgboost as xgb
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import warnings

warnings.filterwarnings('ignore')

try:
    from utils import load_hyperparams, load_backward_gmm_params, get_default_hyperparams
except ImportError:
    import sys
    sys.path.insert(0, os.getcwd())
    from utils import load_hyperparams, load_backward_gmm_params, get_default_hyperparams

INPUT_COLS = ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti']
CAST_PATH = 'cleaned_cast_dataset.csv'
OUTPUT_PATH = 'synthetic_cast.csv'
N_SAMPLES = 50000

In [2]:
# Load cast data (composition from Alloy Name + target columns)
def load_cast(path):
    try:
        df = pd.read_csv(path, encoding='utf-8')
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding='latin-1')
    def extract(text):
        if pd.isna(text): return {}
        text = str(text).replace('AI', 'Al')
        m = re.search(r'\((.*?)\)', text)
        if not m: return {}
        f = m.group(1)
        comp = {}
        for el in INPUT_COLS:
            if el == 'Al': continue
            p = re.compile(rf"{el}(\d*\.?\d*)")
            h = p.search(f)
            if h:
                v = h.group(1)
                comp[el] = float(v) if (v and v != '.') else 0.5
        t = sum(comp.values())
        comp['Al'] = 100 - t if 0 < t < 100 else 0.0
        return comp
    ext = df['Alloy Name'].apply(extract)
    comp_df = pd.DataFrame(list(ext)).fillna(0.0)
    for c in INPUT_COLS:
        if c not in comp_df.columns: comp_df[c] = 0.0
    df = pd.concat([df, comp_df], axis=1)
    df = df[df['Al'] > 1.0].reset_index(drop=True)
    exclude = INPUT_COLS + ['Series', 'Parent Alloy', 'Alloy Name', 'AlloyNumber', 'Temper', 'Standard']
    targets = [c for c in df.columns if c not in exclude]
    def clean_val(val):
        if pd.isna(val): return np.nan
        s = str(val).strip()
        if '-' in s:
            try:
                parts = [re.sub(r'[^0-9.]', '', p) for p in s.split('-')]
                nums = [float(p) for p in parts if p]
                return sum(nums)/len(nums) if nums else np.nan
            except: pass
        m = re.search(r"[-+]?[0-9]*\.[0-9]+|[0-9]+", s)
        return float(m.group()) if m else np.nan
    for col in targets:
        df[col] = df[col].apply(clean_val)
    return df, targets

df_real, target_list = load_cast(CAST_PATH) if os.path.exists(CAST_PATH) else (None, [])
if df_real is None:
    raise FileNotFoundError(f'Not found: {CAST_PATH}. Run with cast data.')
print(f'Loaded cast data: {df_real.shape}, targets: {len(target_list)}')

Loaded cast data: (41, 44), targets: 30


In [3]:
# Fit GMM on real cast compositions and sample
gmm_params = load_backward_gmm_params('cast') or get_default_hyperparams('GMM')
X_real = df_real[INPUT_COLS]
gmm = GaussianMixture(**gmm_params)
gmm.fit(X_real)
samples, _ = gmm.sample(N_SAMPLES)
gen_df = pd.DataFrame(samples, columns=INPUT_COLS)
gen_df[gen_df < 0] = 0
gen_df['_sum'] = gen_df.sum(axis=1)
gen_df = gen_df[gen_df['_sum'] > 0.1]
for c in INPUT_COLS:
    gen_df[c] = (gen_df[c] / gen_df['_sum']) * 100
gen_df = gen_df.drop(columns=['_sum']).fillna(0.0).reset_index(drop=True)
print(f'Sampled {len(gen_df)} valid compositions.')

Sampled 50000 valid compositions.


In [4]:
# Build per-target models from config, train on real data, predict on synthetic
def build_model(name, params):
    p = params.copy() if params else get_default_hyperparams(name)
    if name == 'XGBoost':
        return xgb.XGBRegressor(objective='reg:squarederror', **{k: v for k, v in p.items()})
    if name == 'RandomForest':
        return RandomForestRegressor(**{k: v for k, v in p.items()})
    if name == 'GradientBoosting':
        return GradientBoostingRegressor(**{k: v for k, v in p.items()})
    return None

cast_config = load_hyperparams('cast')
by_target = (cast_config or {}).get('by_target') or {}
if not by_target:
    raise ValueError('cast.by_target not in config. Run 01_hyperparameter_tuning_forward.ipynb first.')

for target, spec in by_target.items():
    if target not in df_real.columns or df_real[target].notna().sum() < 10:
        continue
    model_name = spec.get('model')
    params = spec.get('params')
    model = build_model(model_name, params)
    if model is None:
        continue
    df_t = df_real.dropna(subset=[target])
    X_train = df_t[INPUT_COLS]
    y_train = df_t[target]
    model.fit(X_train, y_train)
    gen_df[target] = model.predict(gen_df[INPUT_COLS])
    print(f'  Predicted {target[:45]}')

print(f'Total columns: {list(gen_df.columns)}')

  Predicted Elastic Modulus (GPa)


  Predicted Fatigue Strength (MPa)


  Predicted Shear Modulus (GPa)


  Predicted Tensile Strength: Ultimate (UTS) (MPa)


  Predicted Tensile Strength: Yield (Proof) (MPa)


  Predicted Thermal Conductivity (W/m-K)


  Predicted Electrical Conductivity: Equal Volume (% IACS


  Predicted Electrical Conductivity: Equal Weight (% IACS


  Predicted Strength to Weight: Axial (points)
Total columns: ['Al', 'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Ni', 'Zn', 'Ga', 'V', 'Ti', 'Elastic Modulus (GPa)', 'Fatigue Strength (MPa)', 'Shear Modulus (GPa)', 'Tensile Strength: Ultimate (UTS) (MPa)', 'Tensile Strength: Yield (Proof) (MPa)', 'Thermal Conductivity (W/m-K)', 'Electrical Conductivity: Equal Volume (% IACS)', 'Electrical Conductivity: Equal Weight (% IACS)', 'Strength to Weight: Axial (points)']


In [5]:
# Save synthetic pool for backward search
gen_df.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {len(gen_df)} rows to {OUTPUT_PATH}')

Saved 50000 rows to synthetic_cast.csv
